# Step 6 — Sklearn Pipeline with ColumnTransformer

Wrap preprocessing + model into **one object** so `.fit()` and `.predict()` handle everything.

| Component | Choice | Why |
|-----------|--------|-----|
| Numeric scaler | `RobustScaler` | Resistant to outliers (uses median/IQR instead of mean/std) |
| Categorical encoder | `OneHotEncoder` | No ordinal assumption; sparse-friendly |
| Model | `LogisticRegression` | Fast baseline with `class_weight='balanced'` |
| Orchestration | `Pipeline` + `ColumnTransformer` | Single object, no leakage, serialisable |

In [ ]:
import pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, roc_auc_score

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120})
SEED = 42
print('Imports OK')

---
## 1 · Load & Split Data

In [ ]:
df = pd.read_csv(pathlib.Path('..') / 'data' / 'raw' / 'credit_risk_dataset.csv')
TARGET = 'loan_status'

X = df.drop(columns=[TARGET])
y = df[TARGET]

# Identify column types
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

print(f'Numeric  ({len(num_cols)}): {num_cols}')
print(f'Categor. ({len(cat_cols)}): {cat_cols}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'\nTrain: {X_train.shape}  |  Test: {X_test.shape}')

---
## 2 · Build the Pipeline

```
Pipeline
 +-- ColumnTransformer
 |    +-- numeric:  SimpleImputer(median) -> RobustScaler
 |    +-- categorical: SimpleImputer(most_frequent) -> OneHotEncoder
 +-- LogisticRegression(class_weight='balanced')
```

In [ ]:
# ── Sub-pipelines per column type ────────────────────────────────────
numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  RobustScaler()),
])

categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# ── ColumnTransformer ────────────────────────────────────────────────
preprocessor = ColumnTransformer([
    ('num', numeric_pipe,     num_cols),
    ('cat', categorical_pipe, cat_cols),
])

# ── Full pipeline ────────────────────────────────────────────────────
pipe = Pipeline([
    ('preprocess', preprocessor),
    ('model',      LogisticRegression(
                       max_iter=1000,
                       random_state=SEED,
                       class_weight='balanced',
                   )),
])

print(pipe)

### Why this matters

- **No leakage:** Imputer & scaler are `.fit()` on training data only, then `.transform()` on test.
- **One object:** `pipe.fit(X_train, y_train)` does everything. No manual transform steps.
- **Serialisable:** `joblib.dump(pipe, 'model.pkl')` saves the full pipeline for production.
- **RobustScaler** uses median & IQR → not distorted by outliers we saw in EDA (e.g. extreme ages/incomes).

---
## 3 · Fit & Evaluate

In [ ]:
pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(
    pipe, X_test, y_test,
    display_labels=['No Default', 'Default'],
    cmap='Blues', values_format=',', ax=ax,
)
ax.set_title('Pipeline — Confusion Matrix')
plt.tight_layout()
plt.savefig('12_pipeline_cm.png', bbox_inches='tight')
plt.show()

---
## 4 · Inspect Transformed Features

In [ ]:
# Get feature names after transformation
ohe_names = pipe['preprocess'].named_transformers_['cat'] \
                .named_steps['onehot'].get_feature_names_out(cat_cols).tolist()
all_feature_names = num_cols + ohe_names

print(f'Total features after OHE: {len(all_feature_names)}')
print(f'  Numeric:     {len(num_cols)}')
print(f'  OHE columns: {len(ohe_names)}')
print(f'\nOHE columns: {ohe_names}')

In [ ]:
# Show model coefficients
coefs = pd.Series(
    pipe['model'].coef_[0], index=all_feature_names
).sort_values()

fig, ax = plt.subplots(figsize=(8, max(4, len(coefs) * 0.35)))
colours = ['#e74c3c' if v > 0 else '#3498db' for v in coefs.values]
ax.barh(coefs.index, coefs.values, color=colours, edgecolor='white')
ax.axvline(0, color='grey', lw=0.8)
ax.set_xlabel('Coefficient (log-odds)')
ax.set_title('Logistic Regression Coefficients (Pipeline)')
plt.tight_layout()
plt.savefig('13_pipeline_coefs.png', bbox_inches='tight')
plt.show()

---
## 5 · Cross-Validation with the Pipeline

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_scores = cross_validate(
    pipe, X, y, cv=cv,
    scoring=['f1', 'recall', 'precision', 'roc_auc'],
    return_train_score=False,
)

print('5-Fold Stratified CV results:')
for metric in ['f1', 'recall', 'precision', 'roc_auc']:
    vals = cv_scores[f'test_{metric}']
    print(f'  {metric:12s}  {vals.mean():.4f} +/- {vals.std():.4f}')

---
## 6 · Save the Pipeline

In [ ]:
import joblib, os

model_dir = pathlib.Path('..') / 'models'
model_dir.mkdir(exist_ok=True)

model_path = model_dir / 'lr_pipeline_v1.pkl'
joblib.dump(pipe, model_path)

size_kb = os.path.getsize(model_path) / 1024
print(f'Pipeline saved -> {model_path}  ({size_kb:.1f} KB)')

# Quick sanity check: reload and predict
pipe_loaded = joblib.load(model_path)
assert (pipe_loaded.predict(X_test) == y_pred).all()
print('Reload & predict sanity check passed')

---
## Summary

| What | Detail |
|------|--------|
| **Pipeline** | `ColumnTransformer` (impute + scale numerics, impute + OHE categoricals) -> `LogisticRegression` |
| **Scaler** | `RobustScaler` — uses median/IQR, robust to outliers |
| **Encoder** | `OneHotEncoder(handle_unknown='ignore')` — safe for unseen categories at inference |
| **Class balance** | `class_weight='balanced'` carried forward from Step 5 |
| **Serialisation** | Full pipeline saved to `models/lr_pipeline_v1.pkl` |

**Next step →** Swap in stronger models (Random Forest, XGBoost) using the same pipeline structure.